# 04 高阶建模（Phase 4）

这个 notebook 对应 `Agent.md` 里的 **Phase 4: Advanced Modeling**。当前版本不再只做 “tree vs logistic” 单次比较，而是保留两套数据口径，形成四模型矩阵：

- `logistic + traditional_core`
- `logistic + traditional_plus_proxy`
- `advanced + traditional_core`
- `advanced + traditional_plus_proxy`

因此 Phase 4 必须同时报告两类 uplift：

- **模型增量**：同一 feature set 下，`advanced - logistic`
- **数据增量**：同一模型家族下，`traditional_plus_proxy - traditional_core`


## 1. 运行参数与辅助函数

这里要求两个 feature set 共用同一个 optional backend 和同一组 `MODEL_KWARGS`。不允许一套用 LightGBM、另一套用 XGBoost。


In [ ]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse

root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "pyproject.toml").exists()),
    Path.cwd(),
)
src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

warnings.filterwarnings(
    "ignore",
    message="Pandas requires version '1.3.6' or newer of 'bottleneck'",
)

from credit_visable.config import load_settings
from credit_visable.features import FEATURE_SET_NAMES as SUPPORTED_FEATURE_SET_NAMES
from credit_visable.modeling import (
    build_binary_diagnostic_curves,
    evaluate_binary_classifier,
    get_tree_backend_availability,
    train_tree_model,
)
from credit_visable.utils import apply_report_style, get_paths

settings = load_settings()
paths = get_paths()
apply_report_style()

FEATURE_SET_NAMES = list(SUPPORTED_FEATURE_SET_NAMES)
PREPROCESSING_OUTPUT_ROOT = paths.data_processed / "preprocessing"
BASELINE_OUTPUT_ROOT = paths.data_processed / "modeling_baseline"
MODELING_OUTPUT_ROOT = paths.data_processed / "modeling_advanced"
FIGURE_OUTPUT_DIR = paths.reports_figures

OPERATING_THRESHOLD = 0.50
PHASE_4_THRESHOLDS = [0.30, 0.50, 0.70]
MODEL_NAME = "xgboost"
MODEL_KWARGS = {}
RANDOM_STATE = settings.random_state

PHASE_4_READY = False
BACKEND_READY = False
phase4_load_ok = False
selected_backend = None
figure_paths = {}
artifacts_by_set = {}
advanced_outputs_by_set = {}
comparison_metrics = pd.DataFrame()
model_uplift_summary = pd.DataFrame()
data_uplift_summary = pd.DataFrame()
threshold_scenarios = pd.DataFrame()
validation_score_comparison = pd.DataFrame()


def build_preprocessing_files(output_dir: Path) -> dict[str, Path]:
    return {
        "X_train": output_dir / "X_train.npz",
        "X_valid": output_dir / "X_valid.npz",
        "train_meta": output_dir / "train_meta.csv",
        "valid_meta": output_dir / "valid_meta.csv",
        "feature_names": output_dir / "feature_names.csv",
        "manifest": output_dir / "manifest.json",
        "feature_set_manifest": output_dir / "feature_set_manifest.json",
    }


def build_baseline_files(output_dir: Path) -> dict[str, Path]:
    return {
        "metrics": output_dir / "metrics.json",
        "validation_scores": output_dir / "validation_scores.csv",
    }


def _safe_mean(values: np.ndarray) -> float | None:
    if values.size == 0:
        return None
    return float(np.mean(values))


def _matrix_density(matrix: sparse.spmatrix) -> float:
    total_cells = matrix.shape[0] * matrix.shape[1]
    if total_cells == 0:
        return 0.0
    return float(matrix.nnz / total_cells)


def _to_builtin(value):
    if isinstance(value, dict):
        return {str(key): _to_builtin(item) for key, item in value.items()}
    if isinstance(value, list):
        return [_to_builtin(item) for item in value]
    if isinstance(value, tuple):
        return [_to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return [_to_builtin(item) for item in value.tolist()]
    if isinstance(value, (np.floating, float)):
        if not np.isfinite(value):
            return None
        return float(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if pd.isna(value):
        return None
    return value


def build_threshold_scenarios(
    y_true: np.ndarray,
    y_score: np.ndarray,
    feature_set_name: str,
    model_family: str,
) -> pd.DataFrame:
    y_true_array = np.asarray(y_true)
    y_score_array = np.asarray(y_score)
    total_bad = int(y_true_array.sum())
    rows = []

    for threshold in PHASE_4_THRESHOLDS:
        scenario_metrics = evaluate_binary_classifier(
            y_true_array,
            y_score_array,
            threshold=threshold,
        )
        approved_mask = y_score_array < threshold
        rejected_mask = ~approved_mask
        rows.append(
            {
                "feature_set": feature_set_name,
                "model_family": model_family,
                "model_label": f"{model_family}_{feature_set_name}",
                "threshold": float(threshold),
                "approval_rate": float(approved_mask.mean()),
                "reject_rate": float(rejected_mask.mean()),
                "approved_population_bad_rate": _safe_mean(y_true_array[approved_mask]),
                "rejected_population_bad_capture_rate": (
                    float(y_true_array[rejected_mask].sum() / total_bad)
                    if total_bad > 0
                    else 0.0
                ),
                "precision": scenario_metrics["precision"],
                "recall": scenario_metrics["recall"],
                "f1": scenario_metrics["f1"],
            }
        )

    return pd.DataFrame(rows)


def build_feature_importance_frame(model, feature_name_list: list[str]) -> pd.DataFrame | None:
    if not hasattr(model, "feature_importances_"):
        return None

    importance_array = np.asarray(model.feature_importances_, dtype=float)
    if importance_array.size != len(feature_name_list):
        return None

    frame = pd.DataFrame(
        {
            "feature_name": feature_name_list,
            "importance": importance_array,
        }
    )
    return frame.sort_values("importance", ascending=False).reset_index(drop=True)


print(f"Preprocessing output root: {PREPROCESSING_OUTPUT_ROOT}")
print(f"Baseline output root: {BASELINE_OUTPUT_ROOT}")
print(f"Advanced output root: {MODELING_OUTPUT_ROOT}")
print(f"Figure output dir: {FIGURE_OUTPUT_DIR}")
print(f"Feature sets: {FEATURE_SET_NAMES}")
print(f"Requested advanced backend: {MODEL_NAME}")


## 2. Readiness gate

Phase 4 必须同时依赖两套 Phase 2 artifact 和两套 Phase 3 baseline 结果。缺任何一臂，就不能进入 advanced comparison。


In [ ]:
required_preprocessing_files = {
    feature_set_name: build_preprocessing_files(PREPROCESSING_OUTPUT_ROOT / feature_set_name)
    for feature_set_name in FEATURE_SET_NAMES
}
required_baseline_files = {
    feature_set_name: build_baseline_files(BASELINE_OUTPUT_ROOT / feature_set_name)
    for feature_set_name in FEATURE_SET_NAMES
}

readiness_rows = []
for feature_set_name in FEATURE_SET_NAMES:
    readiness_rows.extend(
        [
            {
                "feature_set": feature_set_name,
                "stage": "phase2_preprocessing",
                "artifact": artifact_name,
                "path": str(path),
                "exists": path.exists(),
            }
            for artifact_name, path in required_preprocessing_files[feature_set_name].items()
        ]
    )
    readiness_rows.extend(
        [
            {
                "feature_set": feature_set_name,
                "stage": "phase3_baseline",
                "artifact": artifact_name,
                "path": str(path),
                "exists": path.exists(),
            }
            for artifact_name, path in required_baseline_files[feature_set_name].items()
        ]
    )

readiness_frame = pd.DataFrame(readiness_rows)
PHASE_4_READY = bool(readiness_frame["exists"].all())
display(readiness_frame)
print(f"Phase 4 ready: {PHASE_4_READY}")

if not PHASE_4_READY:
    missing_files = readiness_frame.loc[~readiness_frame["exists"]]
    display(missing_files)
    print("Phase 4 blocked. 请先完成 notebooks/02_preprocessing.ipynb 与 notebooks/03_modeling_baseline.ipynb 的双 feature-set 版本。")


## 3. Backend gate

这里仍然执行 dependency gate，但本 notebook 默认固定使用 `xgboost`，并把同一 backend 同时用于两个 feature set。


In [ ]:
backend_availability = get_tree_backend_availability()
backend_summary = pd.DataFrame(
    [
        {
            "backend": backend_name,
            "installed": details["installed"],
            "install_hint": details["install_hint"],
        }
        for backend_name, details in backend_availability["backends"].items()
    ]
)
display(backend_summary)

supported_backends = list(backend_availability["backends"])
requested_backend = MODEL_NAME.lower() if isinstance(MODEL_NAME, str) and MODEL_NAME else None
BACKEND_READY = False
selected_backend = None

if requested_backend is not None and requested_backend not in supported_backends:
    print(f"Unsupported MODEL_NAME: {MODEL_NAME}. Supported backends: {supported_backends}")
elif requested_backend is not None:
    selected_backend = requested_backend
    BACKEND_READY = backend_availability["backends"][requested_backend]["installed"]
    if not BACKEND_READY:
        print(
            f"Requested backend '{requested_backend}' is not installed. "
            f"{backend_availability['backends'][requested_backend]['install_hint']}"
        )
else:
    selected_backend = backend_availability["preferred_backend"]
    BACKEND_READY = selected_backend is not None
    if not BACKEND_READY:
        print(backend_availability["install_hint"])

if BACKEND_READY:
    print(f"Phase 4 selected backend: {selected_backend}")
else:
    print("Phase 4 blocked until xgboost is available.")


## 4. 加载并校验双 feature-set 输入

这里除了验证每套 preprocessing artifact，还要确认 baseline validation scores 能与当前 validation 主键一一对齐。


In [ ]:
if not PHASE_4_READY:
    print("加载步骤已跳过，因为 Phase 2 / Phase 3 产物不完整。")
elif not BACKEND_READY:
    print("加载步骤已跳过，因为没有可用的 tree-model backend。")
else:
    load_summary_rows = []
    validation_errors = []
    reference_train_meta = None
    reference_valid_meta = None

    for feature_set_name in FEATURE_SET_NAMES:
        preprocessing_files = required_preprocessing_files[feature_set_name]
        baseline_files = required_baseline_files[feature_set_name]

        X_train = sparse.load_npz(preprocessing_files["X_train"])
        X_valid = sparse.load_npz(preprocessing_files["X_valid"])
        train_meta = pd.read_csv(preprocessing_files["train_meta"])
        valid_meta = pd.read_csv(preprocessing_files["valid_meta"])
        feature_names = pd.read_csv(preprocessing_files["feature_names"])
        manifest = json.loads(preprocessing_files["manifest"].read_text(encoding="utf-8"))
        feature_set_manifest = json.loads(
            preprocessing_files["feature_set_manifest"].read_text(encoding="utf-8")
        )
        baseline_metrics_payload = json.loads(
            baseline_files["metrics"].read_text(encoding="utf-8")
        )
        baseline_validation_scores = pd.read_csv(baseline_files["validation_scores"])

        feature_name_column = (
            "feature_name" if "feature_name" in feature_names.columns else feature_names.columns[0]
        )
        feature_name_list = feature_names[feature_name_column].astype(str).tolist()

        if settings.target_column not in train_meta.columns:
            validation_errors.append(f"{feature_set_name}: train_meta 缺少目标列")
        if settings.target_column not in valid_meta.columns:
            validation_errors.append(f"{feature_set_name}: valid_meta 缺少目标列")
        if settings.id_column not in train_meta.columns:
            validation_errors.append(f"{feature_set_name}: train_meta 缺少主键列")
        if settings.id_column not in valid_meta.columns:
            validation_errors.append(f"{feature_set_name}: valid_meta 缺少主键列")
        if X_train.shape[0] != len(train_meta):
            validation_errors.append(f"{feature_set_name}: X_train 行数与 train_meta 行数不一致")
        if X_valid.shape[0] != len(valid_meta):
            validation_errors.append(f"{feature_set_name}: X_valid 行数与 valid_meta 行数不一致")
        if X_train.shape[1] == 0 or X_valid.shape[1] == 0:
            validation_errors.append(f"{feature_set_name}: 没有可训练特征列")
        if len(feature_name_list) != X_train.shape[1]:
            validation_errors.append(f"{feature_set_name}: feature_names 条数与矩阵列数不一致")
        if manifest.get("train_shape") != list(X_train.shape):
            validation_errors.append(f"{feature_set_name}: manifest 中 train_shape 与 X_train 不一致")
        if manifest.get("valid_shape") != list(X_valid.shape):
            validation_errors.append(f"{feature_set_name}: manifest 中 valid_shape 与 X_valid 不一致")
        if feature_set_manifest.get("feature_set_name") != feature_set_name:
            validation_errors.append(f"{feature_set_name}: feature_set_manifest 的 feature_set_name 不一致")
        if baseline_metrics_payload.get("model_type") != "logistic_regression_baseline":
            validation_errors.append(f"{feature_set_name}: Phase 3 metrics.json 不是 logistic baseline 结果")
        baseline_required_columns = {settings.id_column, "predicted_pd"}
        missing_baseline_columns = baseline_required_columns - set(baseline_validation_scores.columns)
        if missing_baseline_columns:
            validation_errors.append(
                f"{feature_set_name}: baseline validation_scores 缺少列 {sorted(missing_baseline_columns)}"
            )
        if settings.id_column in baseline_validation_scores.columns and baseline_validation_scores[settings.id_column].duplicated().any():
            validation_errors.append(f"{feature_set_name}: baseline validation_scores 主键不是唯一值")

        sorted_train_meta = train_meta[[settings.id_column, settings.target_column]].sort_values(settings.id_column).reset_index(drop=True)
        sorted_valid_meta = valid_meta[[settings.id_column, settings.target_column]].sort_values(settings.id_column).reset_index(drop=True)
        if reference_train_meta is None:
            reference_train_meta = sorted_train_meta
            reference_valid_meta = sorted_valid_meta
        else:
            if not reference_train_meta.equals(sorted_train_meta):
                validation_errors.append(f"{feature_set_name}: train_meta 与参考 feature set 的主键/目标不一致")
            if not reference_valid_meta.equals(sorted_valid_meta):
                validation_errors.append(f"{feature_set_name}: valid_meta 与参考 feature set 的主键/目标不一致")

        baseline_merge_preview = valid_meta[[settings.id_column, settings.target_column]].merge(
            baseline_validation_scores[[settings.id_column, "predicted_pd"]],
            on=settings.id_column,
            how="left",
        )
        if baseline_merge_preview["predicted_pd"].isna().any():
            validation_errors.append(f"{feature_set_name}: baseline validation_scores 无法覆盖全部 validation 样本")

        artifacts_by_set[feature_set_name] = {
            "X_train": X_train,
            "X_valid": X_valid,
            "train_meta": train_meta,
            "valid_meta": valid_meta,
            "feature_names": feature_name_list,
            "manifest": manifest,
            "feature_set_manifest": feature_set_manifest,
            "baseline_metrics_payload": baseline_metrics_payload,
            "baseline_validation_scores": baseline_merge_preview,
        }
        load_summary_rows.extend(
            [
                {
                    "feature_set": feature_set_name,
                    "artifact": "X_train",
                    "shape": X_train.shape,
                    "density": _matrix_density(X_train),
                },
                {
                    "feature_set": feature_set_name,
                    "artifact": "X_valid",
                    "shape": X_valid.shape,
                    "density": _matrix_density(X_valid),
                },
                {
                    "feature_set": feature_set_name,
                    "artifact": "baseline_validation_scores",
                    "shape": baseline_merge_preview.shape,
                    "density": None,
                },
            ]
        )

    phase4_load_ok = not validation_errors
    display(pd.DataFrame(load_summary_rows))
    if validation_errors:
        display(pd.DataFrame({"validation_error": validation_errors}))
        print("Phase 4 stopped because the saved preprocessing or baseline artifacts failed validation.")
    else:
        print("Phase 4 load validation passed for both feature sets.")


## 5. 训练 advanced model，并拆分模型增量与数据增量

这里先对每个 feature set 训练同一 backend 的 advanced model，再统一汇总成四模型 comparison matrix。


In [ ]:
if not PHASE_4_READY:
    print("训练与对比已跳过，因为 Phase 2 / Phase 3 产物不完整。")
elif not BACKEND_READY:
    print("训练与对比已跳过，因为没有可用的 tree-model backend。")
elif not phase4_load_ok:
    print("训练与对比已跳过，因为加载校验未通过。")
else:
    comparison_rows = []
    model_uplift_rows = []
    threshold_frames = []

    for feature_set_name in FEATURE_SET_NAMES:
        bundle = artifacts_by_set[feature_set_name]
        y_train = bundle["train_meta"][settings.target_column]
        y_valid = bundle["valid_meta"][settings.target_column].to_numpy()
        baseline_y_score = bundle["baseline_validation_scores"]["predicted_pd"].to_numpy()

        baseline_primary_metrics = evaluate_binary_classifier(
            y_valid,
            baseline_y_score,
            threshold=OPERATING_THRESHOLD,
        )
        baseline_diagnostic_curves = build_binary_diagnostic_curves(y_valid, baseline_y_score)
        threshold_frames.append(
            build_threshold_scenarios(y_valid, baseline_y_score, feature_set_name, "logistic")
        )

        advanced_model = train_tree_model(
            bundle["X_train"],
            y_train,
            model_name=selected_backend,
            random_state=RANDOM_STATE,
            **MODEL_KWARGS,
        )
        advanced_y_score = advanced_model.predict_proba(bundle["X_valid"])[:, 1]
        advanced_primary_metrics = evaluate_binary_classifier(
            y_valid,
            advanced_y_score,
            threshold=OPERATING_THRESHOLD,
        )
        advanced_diagnostic_curves = build_binary_diagnostic_curves(y_valid, advanced_y_score)
        threshold_frames.append(
            build_threshold_scenarios(y_valid, advanced_y_score, feature_set_name, "advanced")
        )
        feature_importance = build_feature_importance_frame(
            advanced_model,
            bundle["feature_names"],
        )

        advanced_validation_scores = bundle["baseline_validation_scores"].rename(
            columns={"predicted_pd": "logistic_predicted_pd"}
        ).copy()
        advanced_validation_scores["advanced_predicted_pd"] = advanced_y_score
        advanced_validation_scores["feature_set"] = feature_set_name
        advanced_validation_scores["logistic_decision_at_050"] = np.where(
            advanced_validation_scores["logistic_predicted_pd"] >= OPERATING_THRESHOLD,
            "reject",
            "approve",
        )
        advanced_validation_scores["advanced_decision_at_050"] = np.where(
            advanced_validation_scores["advanced_predicted_pd"] >= OPERATING_THRESHOLD,
            "reject",
            "approve",
        )
        advanced_validation_scores["pd_delta_advanced_minus_logistic"] = (
            advanced_validation_scores["advanced_predicted_pd"]
            - advanced_validation_scores["logistic_predicted_pd"]
        )

        advanced_outputs_by_set[feature_set_name] = {
            "advanced_model": advanced_model,
            "advanced_primary_metrics": advanced_primary_metrics,
            "advanced_diagnostic_curves": advanced_diagnostic_curves,
            "baseline_primary_metrics": baseline_primary_metrics,
            "baseline_diagnostic_curves": baseline_diagnostic_curves,
            "feature_importance": feature_importance,
            "validation_scores": advanced_validation_scores,
        }

        comparison_rows.extend(
            [
                {
                    "feature_set": feature_set_name,
                    "model_family": "logistic",
                    "model_label": f"logistic_{feature_set_name}",
                    "feature_count": len(bundle["feature_names"]),
                    "roc_auc": baseline_primary_metrics["roc_auc"],
                    "average_precision": baseline_primary_metrics["average_precision"],
                    "accuracy": baseline_primary_metrics["accuracy"],
                    "precision": baseline_primary_metrics["precision"],
                    "recall": baseline_primary_metrics["recall"],
                    "f1": baseline_primary_metrics["f1"],
                    "ks_statistic": baseline_diagnostic_curves["ks"]["statistic"],
                    "ks_threshold": baseline_diagnostic_curves["ks"]["threshold"],
                },
                {
                    "feature_set": feature_set_name,
                    "model_family": "advanced",
                    "model_label": f"{selected_backend}_{feature_set_name}",
                    "feature_count": len(bundle["feature_names"]),
                    "roc_auc": advanced_primary_metrics["roc_auc"],
                    "average_precision": advanced_primary_metrics["average_precision"],
                    "accuracy": advanced_primary_metrics["accuracy"],
                    "precision": advanced_primary_metrics["precision"],
                    "recall": advanced_primary_metrics["recall"],
                    "f1": advanced_primary_metrics["f1"],
                    "ks_statistic": advanced_diagnostic_curves["ks"]["statistic"],
                    "ks_threshold": advanced_diagnostic_curves["ks"]["threshold"],
                },
            ]
        )
        model_uplift_rows.append(
            {
                "feature_set": feature_set_name,
                "comparison": f"{selected_backend}_minus_logistic",
                "roc_auc_delta": float(advanced_primary_metrics["roc_auc"] - baseline_primary_metrics["roc_auc"]),
                "average_precision_delta": float(advanced_primary_metrics["average_precision"] - baseline_primary_metrics["average_precision"]),
                "accuracy_delta": float(advanced_primary_metrics["accuracy"] - baseline_primary_metrics["accuracy"]),
                "precision_delta": float(advanced_primary_metrics["precision"] - baseline_primary_metrics["precision"]),
                "recall_delta": float(advanced_primary_metrics["recall"] - baseline_primary_metrics["recall"]),
                "f1_delta": float(advanced_primary_metrics["f1"] - baseline_primary_metrics["f1"]),
                "ks_statistic_delta": float(
                    advanced_diagnostic_curves["ks"]["statistic"] - baseline_diagnostic_curves["ks"]["statistic"]
                ),
            }
        )

    comparison_metrics = pd.DataFrame(comparison_rows)
    model_uplift_summary = pd.DataFrame(model_uplift_rows)
    threshold_scenarios = pd.concat(threshold_frames, ignore_index=True)

    data_uplift_rows = []
    metric_columns = [
        "feature_count",
        "roc_auc",
        "average_precision",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "ks_statistic",
    ]
    threshold_metric_columns = [
        "approval_rate",
        "reject_rate",
        "approved_population_bad_rate",
        "rejected_population_bad_capture_rate",
    ]
    operating_threshold_rows = threshold_scenarios.loc[
        threshold_scenarios["threshold"] == OPERATING_THRESHOLD
    ].reset_index(drop=True)

    for model_family in ["logistic", "advanced"]:
        core_row = comparison_metrics.loc[
            (comparison_metrics["feature_set"] == "traditional_core")
            & (comparison_metrics["model_family"] == model_family)
        ].iloc[0]
        proxy_row = comparison_metrics.loc[
            (comparison_metrics["feature_set"] == "traditional_plus_proxy")
            & (comparison_metrics["model_family"] == model_family)
        ].iloc[0]
        core_operating_row = operating_threshold_rows.loc[
            (operating_threshold_rows["feature_set"] == "traditional_core")
            & (operating_threshold_rows["model_family"] == model_family)
        ].iloc[0]
        proxy_operating_row = operating_threshold_rows.loc[
            (operating_threshold_rows["feature_set"] == "traditional_plus_proxy")
            & (operating_threshold_rows["model_family"] == model_family)
        ].iloc[0]
        data_uplift_rows.append(
            {
                "model_family": model_family,
                "comparison": "traditional_plus_proxy_minus_traditional_core",
                **{
                    f"{column}_delta": float(proxy_row[column] - core_row[column])
                    for column in metric_columns
                },
                **{
                    f"{column}_delta": float(proxy_operating_row[column] - core_operating_row[column])
                    for column in threshold_metric_columns
                },
            }
        )

    data_uplift_summary = pd.DataFrame(data_uplift_rows)

    validation_score_comparison = artifacts_by_set[FEATURE_SET_NAMES[0]]["valid_meta"][
        [settings.id_column, settings.target_column]
    ].copy()
    for feature_set_name in FEATURE_SET_NAMES:
        validation_score_comparison = validation_score_comparison.merge(
            advanced_outputs_by_set[feature_set_name]["validation_scores"][
                [settings.id_column, "logistic_predicted_pd", "advanced_predicted_pd"]
            ].rename(
                columns={
                    "logistic_predicted_pd": f"{feature_set_name}_logistic_predicted_pd",
                    "advanced_predicted_pd": f"{feature_set_name}_advanced_predicted_pd",
                }
            ),
            on=settings.id_column,
            how="left",
        )

    display(comparison_metrics)
    display(model_uplift_summary)
    display(data_uplift_summary)
    display(operating_threshold_rows)


## 6. 可视化与落盘

这里至少输出三张四模型对比图：ROC、PR、KS。若 backend 提供 `feature_importances_`，则额外输出原生特征重要性图，但必须明确它不是 SHAP。


In [ ]:
if not PHASE_4_READY:
    print("图表与落盘已跳过，因为 Phase 2 / Phase 3 产物不完整。")
elif not BACKEND_READY:
    print("图表与落盘已跳过，因为没有可用的 tree-model backend。")
elif not phase4_load_ok:
    print("图表与落盘已跳过，因为加载校验未通过。")
else:
    FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    MODELING_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    figure_paths = {
        "roc_comparison": FIGURE_OUTPUT_DIR / "phase4_four_model_roc_comparison.png",
        "pr_comparison": FIGURE_OUTPUT_DIR / "phase4_four_model_pr_comparison.png",
        "ks_comparison": FIGURE_OUTPUT_DIR / "phase4_four_model_ks_comparison.png",
    }

    fig, ax = plt.subplots(figsize=(6, 5))
    for feature_set_name in FEATURE_SET_NAMES:
        baseline_curves = advanced_outputs_by_set[feature_set_name]["baseline_diagnostic_curves"]
        baseline_metrics = advanced_outputs_by_set[feature_set_name]["baseline_primary_metrics"]
        advanced_curves = advanced_outputs_by_set[feature_set_name]["advanced_diagnostic_curves"]
        advanced_metrics = advanced_outputs_by_set[feature_set_name]["advanced_primary_metrics"]
        ax.plot(
            baseline_curves["roc"]["fpr"],
            baseline_curves["roc"]["tpr"],
            label=f"logistic_{feature_set_name} ROC-AUC = {baseline_metrics['roc_auc']:.4f}",
        )
        ax.plot(
            advanced_curves["roc"]["fpr"],
            advanced_curves["roc"]["tpr"],
            label=f"{selected_backend}_{feature_set_name} ROC-AUC = {advanced_metrics['roc_auc']:.4f}",
        )
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
    ax.set_title("Phase 4 ROC Comparison")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    fig.savefig(figure_paths["roc_comparison"], dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 5))
    for feature_set_name in FEATURE_SET_NAMES:
        baseline_curves = advanced_outputs_by_set[feature_set_name]["baseline_diagnostic_curves"]
        baseline_metrics = advanced_outputs_by_set[feature_set_name]["baseline_primary_metrics"]
        advanced_curves = advanced_outputs_by_set[feature_set_name]["advanced_diagnostic_curves"]
        advanced_metrics = advanced_outputs_by_set[feature_set_name]["advanced_primary_metrics"]
        ax.plot(
            baseline_curves["precision_recall"]["recall"],
            baseline_curves["precision_recall"]["precision"],
            label=f"logistic_{feature_set_name} AP = {baseline_metrics['average_precision']:.4f}",
        )
        ax.plot(
            advanced_curves["precision_recall"]["recall"],
            advanced_curves["precision_recall"]["precision"],
            label=f"{selected_backend}_{feature_set_name} AP = {advanced_metrics['average_precision']:.4f}",
        )
    ax.set_title("Phase 4 Precision-Recall Comparison")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(loc="lower left", fontsize=8)
    fig.tight_layout()
    fig.savefig(figure_paths["pr_comparison"], dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    for feature_set_name in FEATURE_SET_NAMES:
        baseline_curves = advanced_outputs_by_set[feature_set_name]["baseline_diagnostic_curves"]
        advanced_curves = advanced_outputs_by_set[feature_set_name]["advanced_diagnostic_curves"]
        ax.plot(
            baseline_curves["ks"]["thresholds"],
            baseline_curves["ks"]["values"],
            label=f"logistic_{feature_set_name} KS = {baseline_curves['ks']['statistic']:.4f}",
        )
        ax.plot(
            advanced_curves["ks"]["thresholds"],
            advanced_curves["ks"]["values"],
            label=f"{selected_backend}_{feature_set_name} KS = {advanced_curves['ks']['statistic']:.4f}",
        )
    ax.set_title("Phase 4 KS Comparison")
    ax.set_xlabel("Score Threshold")
    ax.set_ylabel("KS Statistic")
    ax.legend(loc="best", fontsize=8)
    fig.tight_layout()
    fig.savefig(figure_paths["ks_comparison"], dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    for feature_set_name in FEATURE_SET_NAMES:
        output_dir = MODELING_OUTPUT_ROOT / feature_set_name
        output_dir.mkdir(parents=True, exist_ok=True)
        threshold_subset = threshold_scenarios.loc[
            threshold_scenarios["feature_set"] == feature_set_name
        ].reset_index(drop=True)
        comparison_subset = comparison_metrics.loc[
            comparison_metrics["feature_set"] == feature_set_name
        ].reset_index(drop=True)
        validation_subset = advanced_outputs_by_set[feature_set_name]["validation_scores"]
        feature_importance = advanced_outputs_by_set[feature_set_name]["feature_importance"]
        metrics_payload = {
            "model_type": f"{selected_backend}_advanced",
            "feature_set": feature_set_name,
            "backend": selected_backend,
            "model_family": "advanced",
            "baseline_primary_metrics": advanced_outputs_by_set[feature_set_name]["baseline_primary_metrics"],
            "advanced_primary_metrics": advanced_outputs_by_set[feature_set_name]["advanced_primary_metrics"],
            "model_uplift": model_uplift_summary.loc[
                model_uplift_summary["feature_set"] == feature_set_name
            ].to_dict(orient="records"),
            "figure_paths": {key: str(path) for key, path in figure_paths.items()},
            "threshold_scenarios": threshold_subset.to_dict(orient="records"),
            "feature_importance_note": (
                "Native feature importance is a backend-specific heuristic and must not be described as SHAP."
            ),
        }

        validation_subset.to_csv(output_dir / "validation_scores.csv", index=False)
        threshold_subset.to_csv(output_dir / "threshold_scenarios.csv", index=False)
        comparison_subset.to_csv(output_dir / "comparison_metrics.csv", index=False)
        joblib.dump(
            advanced_outputs_by_set[feature_set_name]["advanced_model"],
            output_dir / "advanced_model.joblib",
        )
        (output_dir / "metrics.json").write_text(
            json.dumps(_to_builtin(metrics_payload), indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

        if feature_importance is not None and not feature_importance.empty:
            feature_importance.to_csv(output_dir / "feature_importance.csv", index=False)
            feature_figure_path = FIGURE_OUTPUT_DIR / f"phase4_{feature_set_name}_feature_importance.png"
            figure_paths[f"feature_importance_{feature_set_name}"] = feature_figure_path
            top_features = feature_importance.head(20).sort_values("importance", ascending=True)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.barh(top_features["feature_name"], top_features["importance"], color="#4C78A8")
            ax.set_title(f"{selected_backend} native feature importance: {feature_set_name}")
            ax.set_xlabel("importance")
            fig.tight_layout()
            fig.savefig(feature_figure_path, dpi=150, bbox_inches="tight")
            plt.show()
            plt.close(fig)

    comparison_metrics.to_csv(MODELING_OUTPUT_ROOT / "comparison_metrics.csv", index=False)
    model_uplift_summary.to_csv(MODELING_OUTPUT_ROOT / "model_uplift_summary.csv", index=False)
    data_uplift_summary.to_csv(MODELING_OUTPUT_ROOT / "data_uplift_summary.csv", index=False)
    threshold_scenarios.to_csv(MODELING_OUTPUT_ROOT / "threshold_scenarios.csv", index=False)
    validation_score_comparison.to_csv(
        MODELING_OUTPUT_ROOT / "validation_score_comparison.csv",
        index=False,
    )
    summary_payload = {
        "comparison_type": "four_model_matrix",
        "backend": selected_backend,
        "feature_sets": FEATURE_SET_NAMES,
        "operating_threshold": OPERATING_THRESHOLD,
        "comparison_metrics": comparison_metrics.to_dict(orient="records"),
        "model_uplift_summary": model_uplift_summary.to_dict(orient="records"),
        "data_uplift_summary": data_uplift_summary.to_dict(orient="records"),
        "figure_paths": {key: str(path) for key, path in figure_paths.items()},
    }
    (MODELING_OUTPUT_ROOT / "summary.json").write_text(
        json.dumps(_to_builtin(summary_payload), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


## 7. Checkpoint

到这里为止，Phase 4 已经同时回答两个问题：

- 在同一 data regime 下，advanced model 是否比 logistic 更强
- 在同一模型家族下，加入 proxy 是否带来增益

下一步如果进入 Phase 5，必须把 proxy uplift 与 fairness / governance 风险一起解释，而不是直接把 uplift 当成 production gain。


In [ ]:
if not PHASE_4_READY:
    print("Checkpoint: Phase 4 仍被阻断，请先补齐双 feature-set 的 Phase 2 / Phase 3 标准产物。")
elif not BACKEND_READY:
    print("Checkpoint: Phase 4 仍被阻断，请先安装 xgboost。")
elif not phase4_load_ok:
    print("Checkpoint: Phase 4 仍停在加载校验，请先修复 preprocessing 或 baseline artifact 的一致性问题。")
else:
    print("Phase 4 advanced modeling complete.")
    print(f"Advanced backend: {selected_backend}")
    print("模型增量摘要：")
    display(model_uplift_summary)
    print("数据增量摘要：")
    display(data_uplift_summary)
    print(
        "下一步如果进入 Phase 5 explainability / fairness，需要显式审查 proxy-sensitive 变量，而不是把 native feature importance 当成 SHAP。"
    )


In [ ]:
# Phase 4 report upgrade add-ons
from credit_visable.utils import (
    REPORT_COLOR_PALETTE,
    REPORT_LINESTYLES,
    add_conclusion_annotation,
    build_report_summary_fields,
    format_percent_axis,
    to_builtin,
)

if not PHASE_4_READY:
    print("Phase 4 report upgrade skipped because upstream artifacts are incomplete.")
elif not BACKEND_READY:
    print("Phase 4 report upgrade skipped because no tree backend is available.")
elif not phase4_load_ok:
    print("Phase 4 report upgrade skipped because load validation failed.")
else:
    FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    MODELING_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    phase4_extra_figure_paths = {
        "calibration_comparison": FIGURE_OUTPUT_DIR / "phase4_four_model_calibration_comparison.png",
        "gain_comparison": FIGURE_OUTPUT_DIR / "phase4_four_model_gain_comparison.png",
        "lift_comparison": FIGURE_OUTPUT_DIR / "phase4_four_model_lift_comparison.png",
    }
    calibration_frames = []
    gain_frames = []
    lift_frames = []
    metric_rows = []

    for feature_set_name in FEATURE_SET_NAMES:
        metric_rows.append(
            {
                "model_label": f"logistic_{feature_set_name}",
                "brier_score": advanced_outputs_by_set[feature_set_name]["baseline_primary_metrics"]["brier_score"],
                "positive_rate_baseline": advanced_outputs_by_set[feature_set_name]["baseline_primary_metrics"]["positive_rate_baseline"],
            }
        )
        metric_rows.append(
            {
                "model_label": f"{selected_backend}_{feature_set_name}",
                "brier_score": advanced_outputs_by_set[feature_set_name]["advanced_primary_metrics"]["brier_score"],
                "positive_rate_baseline": advanced_outputs_by_set[feature_set_name]["advanced_primary_metrics"]["positive_rate_baseline"],
            }
        )

    metric_frame = pd.DataFrame(metric_rows)
    advanced_candidates = comparison_metrics.loc[comparison_metrics["model_family"] == "advanced"].copy()
    best_candidate = advanced_candidates.sort_values(
        ["roc_auc", "average_precision", "ks_statistic"],
        ascending=[False, False, False],
    ).iloc[0]
    best_model_label = best_candidate["model_label"]

    fig, ax = plt.subplots(figsize=(7, 5))
    for feature_set_name in FEATURE_SET_NAMES:
        for model_family, curve_key, linestyle in [
            ("logistic", "baseline_diagnostic_curves", REPORT_LINESTYLES["baseline"]),
            ("advanced", "advanced_diagnostic_curves", REPORT_LINESTYLES["best"]),
        ]:
            curves = advanced_outputs_by_set[feature_set_name][curve_key]
            calibration_frame = pd.DataFrame(curves["calibration"])
            model_label = f"{selected_backend}_{feature_set_name}" if model_family == "advanced" else f"logistic_{feature_set_name}"
            calibration_frame["model_label"] = model_label
            calibration_frames.append(calibration_frame)
            line_width = 2.6 if model_label == best_model_label else 1.7
            ax.plot(
                calibration_frame["predicted_mean"],
                calibration_frame["observed_rate"],
                linewidth=line_width,
                linestyle=linestyle,
                marker="o",
                label=f"{model_label}",
            )
    ax.plot([0, 1], [0, 1], linestyle="--", color=REPORT_COLOR_PALETTE["neutral"], label="Perfect calibration")
    ax.set_title("Phase 4 Calibration - Four Model Comparison")
    ax.set_xlabel("Mean predicted PD")
    ax.set_ylabel("Observed default rate")
    format_percent_axis(ax, axis="both", decimals=0)
    add_conclusion_annotation(ax, f"Best candidate: {best_model_label}")
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    fig.savefig(phase4_extra_figure_paths["calibration_comparison"], dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    for feature_set_name in FEATURE_SET_NAMES:
        for model_family, curve_key, linestyle in [
            ("logistic", "baseline_diagnostic_curves", REPORT_LINESTYLES["baseline"]),
            ("advanced", "advanced_diagnostic_curves", REPORT_LINESTYLES["best"]),
        ]:
            curves = advanced_outputs_by_set[feature_set_name][curve_key]
            gain_frame = pd.DataFrame(curves["gain"])
            model_label = f"{selected_backend}_{feature_set_name}" if model_family == "advanced" else f"logistic_{feature_set_name}"
            gain_frame["model_label"] = model_label
            gain_frames.append(gain_frame)
            ax.plot(
                gain_frame["population_share"],
                gain_frame["captured_bad_share"],
                linewidth=2.6 if model_label == best_model_label else 1.7,
                linestyle=linestyle,
                label=model_label,
            )
    ax.plot([0, 1], [0, 1], linestyle="--", color=REPORT_COLOR_PALETTE["neutral"], label="Random baseline")
    ax.set_title("Phase 4 Gain Curve - Four Model Comparison")
    ax.set_xlabel("Reviewed population share")
    ax.set_ylabel("Captured bad-share")
    format_percent_axis(ax, axis="both", decimals=0)
    add_conclusion_annotation(ax, "Advanced curves should separate earlier if model uplift is real.")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    fig.savefig(phase4_extra_figure_paths["gain_comparison"], dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    for feature_set_name in FEATURE_SET_NAMES:
        for model_family, curve_key, linestyle in [
            ("logistic", "baseline_diagnostic_curves", REPORT_LINESTYLES["baseline"]),
            ("advanced", "advanced_diagnostic_curves", REPORT_LINESTYLES["best"]),
        ]:
            curves = advanced_outputs_by_set[feature_set_name][curve_key]
            lift_frame = pd.DataFrame(curves["lift"])
            model_label = f"{selected_backend}_{feature_set_name}" if model_family == "advanced" else f"logistic_{feature_set_name}"
            lift_frame["model_label"] = model_label
            lift_frames.append(lift_frame)
            ax.plot(
                lift_frame["population_share"],
                lift_frame["lift"],
                linewidth=2.6 if model_label == best_model_label else 1.7,
                linestyle=linestyle,
                label=model_label,
            )
    ax.axhline(1.0, linestyle="--", color=REPORT_COLOR_PALETTE["neutral"], label="Baseline lift = 1")
    ax.set_title("Phase 4 Lift Curve - Four Model Comparison")
    ax.set_xlabel("Reviewed population share")
    ax.set_ylabel("Lift")
    format_percent_axis(ax, axis="x", decimals=0)
    add_conclusion_annotation(ax, "Higher early lift translates into better review-budget efficiency.")
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    fig.savefig(phase4_extra_figure_paths["lift_comparison"], dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    comparison_metrics = comparison_metrics.drop(columns=["brier_score", "positive_rate_baseline"], errors="ignore")
    comparison_metrics = comparison_metrics.merge(metric_frame, on="model_label", how="left")
    comparison_metrics.to_csv(MODELING_OUTPUT_ROOT / "comparison_metrics.csv", index=False)
    pd.concat(calibration_frames, ignore_index=True).to_csv(MODELING_OUTPUT_ROOT / "calibration_curve_points.csv", index=False)
    pd.concat(gain_frames, ignore_index=True).to_csv(MODELING_OUTPUT_ROOT / "gain_curve_points.csv", index=False)
    pd.concat(lift_frames, ignore_index=True).to_csv(MODELING_OUTPUT_ROOT / "lift_curve_points.csv", index=False)

    best_candidate_summary = {
        "model_label": best_candidate["model_label"],
        "feature_set": best_candidate["feature_set"],
        "model_family": best_candidate["model_family"],
        "roc_auc": float(best_candidate["roc_auc"]),
        "average_precision": float(best_candidate["average_precision"]),
        "ks_statistic": float(best_candidate["ks_statistic"]),
        "brier_score": float(metric_frame.loc[metric_frame["model_label"] == best_model_label, "brier_score"].iloc[0]),
    }
    (MODELING_OUTPUT_ROOT / "best_candidate_summary.json").write_text(
        json.dumps(to_builtin(best_candidate_summary), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    figure_paths.update(phase4_extra_figure_paths)
    summary_path = MODELING_OUTPUT_ROOT / "summary.json"
    existing_summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
    model_uplift_focus = model_uplift_summary.sort_values("roc_auc_delta", ascending=False).iloc[0]
    existing_summary.update(
        build_report_summary_fields(
            headline="Phase 4 now compares four models across ranking, calibration, gain, and lift diagnostics.",
            key_findings=[
                f"Best candidate: {best_model_label}.",
                f"Largest model-uplift feature set: {model_uplift_focus['feature_set']}.",
                f"Top advanced-model ROC-AUC: {best_candidate['roc_auc']:.4f}.",
            ],
            business_implications=[
                "Phase 4 can now separate model uplift from data uplift while also showing calibration quality.",
                "Best-candidate selection is explicit and reusable for Phase 5 explainability and Phase 6 policy analysis.",
                "If the best-ranked model has weak calibration, that gap is now visible before policy translation begins.",
            ],
            figure_paths={**figure_paths, **phase4_extra_figure_paths},
        )
    )
    existing_summary["comparison_metrics"] = comparison_metrics.to_dict(orient="records")
    existing_summary["best_candidate"] = best_candidate_summary
    summary_path.write_text(json.dumps(to_builtin(existing_summary), indent=2, ensure_ascii=False), encoding="utf-8")
